In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT / 'src'))

from data_preprocessing import (
    DEFAULT_FAILURE_EVENTS,
    default_feature_cols,
    label_split_and_save,
    load_metropt3,
    numeric_distribution_summary,
    filter_by_date_range,
)

In [ ]:
DATA_PATH = PROJECT_ROOT / 'dataset' / 'MetroPT3(AirCompressor).csv'
PREPROCESSED_ROOT = PROJECT_ROOT / 'dataset' / 'preprocessed'

# Toggle this to keep or skip an external validation split.
USE_VALIDATION_SPLIT = True

# Explicit split ranges.
TRAIN_START = '2020-02-01 00:00:00'
TRAIN_END = '2020-04-01 00:00:00'
VAL_START = '2020-04-01 00:00:00' if USE_VALIDATION_SPLIT else None
VAL_END = '2020-05-01 00:00:00' if USE_VALIDATION_SPLIT else None
TEST_START = '2020-05-01 00:00:00'
TEST_END = '2020-09-01 00:00:00'

# Plot only a fraction of the rows for a faster anomaly-vs-normal comparison.
PLOT_SAMPLE_FRACTION = 1
PLOT_MAX_FEATURES = 15
PLOT_RANDOM_SEED = 42

FAILURE_EVENTS = DEFAULT_FAILURE_EVENTS

artifacts = label_split_and_save(
    csv_path=DATA_PATH,
    preprocessed_root=PREPROCESSED_ROOT,
    timestamp_col='timestamp',
    failure_events=FAILURE_EVENTS,
    train_start=TRAIN_START,
    train_end=TRAIN_END,
    val_start=VAL_START,
    val_end=VAL_END,
    test_start=TEST_START,
    test_end=TEST_END,
)

PLOTS_DIR = PREPROCESSED_ROOT / 'plots' / artifacts.run_id
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

labeled_df = load_metropt3(artifacts.labeled_csv_path)
labeled_df = filter_by_date_range(labeled_df, timestamp_col='timestamp', data_start=TRAIN_START, data_end=TEST_END)

print(f'Dataset path: {DATA_PATH}')
print(f'Rows in selected range: {len(labeled_df):,}')
print(f'Columns: {labeled_df.shape[1]}')
print('Original CSV was updated in place with failure labels.')
print(f'Preprocessed root: {artifacts.preprocessed_root}')
print(f'Split run directory: {artifacts.split_dir}')
print(f'Train split: {artifacts.train_csv_path}')
if USE_VALIDATION_SPLIT:
    print(f'Val split: {artifacts.val_csv_path}')
else:
    print('Validation split: disabled (no val_split.csv was written).')
print(f'Test split: {artifacts.test_csv_path}')
print(f'Plots directory: {PLOTS_DIR}')

In [ ]:
df = labeled_df
train_df = load_metropt3(artifacts.train_csv_path)
if USE_VALIDATION_SPLIT:
    val_df = load_metropt3(artifacts.val_csv_path)
else:
    val_df = None
test_df = load_metropt3(artifacts.test_csv_path)
feature_cols = default_feature_cols(df, timestamp_col='timestamp')

print(f'Feature count: {len(feature_cols)}')
print('Features:')
for c in feature_cols:
    print(f' - {c}')

print('\nSplit periods:')
print(f' - Train start: {train_df["timestamp"].min()}')
print(f' - Train end:   {train_df["timestamp"].max()}')
if USE_VALIDATION_SPLIT:
    print(f' - Val start:   {val_df["timestamp"].min()}')
    print(f' - Val end:     {val_df["timestamp"].max()}')
else:
    print(' - Validation split: disabled')
print(f' - Test start:  {test_df["timestamp"].min()}')
print(f' - Test end:    {test_df["timestamp"].max()}')
print(f' - Train rows:  {len(train_df):,}')
if USE_VALIDATION_SPLIT:
    print(f' - Val rows:    {len(val_df):,}')
print(f' - Test rows:   {len(test_df):,}')

In [ ]:
def failure_proportion_table(x):
    counts = x['failure_label'].value_counts(dropna=False).sort_index()
    labels = pd.Index([0, 1], dtype='int64')
    counts = counts.reindex(labels, fill_value=0)
    out = pd.DataFrame({
        'count': counts.values,
        'proportion': (counts.values / len(x)) if len(x) else [0.0, 0.0],
    }, index=['non_failure(0)', 'failure(1)'])
    return out

train_failure_prop = failure_proportion_table(train_df)
test_failure_prop = failure_proportion_table(test_df)

print('Train failure/non-failure summary')
display(train_failure_prop)

if USE_VALIDATION_SPLIT and val_df is not None:
    val_failure_prop = failure_proportion_table(val_df)
    print('Val failure/non-failure summary')
    display(val_failure_prop)
else:
    print('Validation split disabled; skipping val failure summary.')

print('Test failure/non-failure summary')
display(test_failure_prop)

In [ ]:
def _sample_for_plotting(x, frac, seed):
    if len(x) == 0:
        return x
    frac = max(0.0, min(1.0, float(frac)))
    if frac >= 1.0:
        return x
    return x.sample(frac=frac, random_state=seed)

def plot_normal_vs_anomaly_distributions(df, feature_cols, sample_frac=0.7, max_features=6, seed=42, output_path=None):
    sampled_parts = []
    for _, group in df.groupby('failure_label'):
        sampled_parts.append(_sample_for_plotting(group, sample_frac, seed))
    plot_df = pd.concat(sampled_parts, ignore_index=True) if sampled_parts else df.copy()
    plot_df = plot_df.assign(label_name=plot_df['failure_label'].map({0: 'normal', 1: 'anomaly'}))
    plot_features = list(feature_cols)[:max_features]
    if len(plot_features) == 0:
        print('No numeric features available for plotting.')
        return None
    n_cols = 2
    n_rows = (len(plot_features) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4 * n_rows))
    axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]
    for ax, feature in zip(axes, plot_features):
        sns.histplot(
            data=plot_df,
            x=feature,
            hue='label_name',
            stat='density',
            common_norm=False,
            bins=40,
            element='step',
            fill=False,
            ax=ax,
        )
        ax.set_title(feature)
        ax.set_xlabel(feature)
        ax.set_ylabel('density')
    for ax in axes[len(plot_features):]:
        ax.axis('off')
    fig.suptitle(f'Normal vs anomaly distributions on {sample_frac:.0%} sampled data', y=1.02)
    plt.tight_layout()
    if output_path is not None:
        fig.savefig(output_path, dpi=160, bbox_inches='tight')
        print(f'Saved plot: {output_path}')
    plt.show()
    return output_path

distribution_plot_path = PLOTS_DIR / 'normal_vs_anomaly_distributions.png'
plot_normal_vs_anomaly_distributions(
    labeled_df,
    feature_cols,
    sample_frac=PLOT_SAMPLE_FRACTION,
    max_features=PLOT_MAX_FEATURES,
    seed=PLOT_RANDOM_SEED,
    output_path=distribution_plot_path,
)

In [ ]:
train_stats = numeric_distribution_summary(train_df, feature_cols)
test_stats = numeric_distribution_summary(test_df, feature_cols)

print('Train distribution stats (min, p10, median, p90, mean, std, max)')
display(train_stats.round(4))

print('Test distribution stats (min, p10, median, p90, mean, std, max)')
display(test_stats.round(4))

In [ ]:
print('Failure label counts (full dataset):')
display(df['failure_label'].value_counts(dropna=False).rename('count').to_frame())

print('Failure windows in use:')
display(pd.DataFrame(FAILURE_EVENTS))